> 在线讲义：[第 26 章 · 单个算子：分块、布局与切分](https://wikieden.github.io/ai-infra-from-constraints/lessons/26-tiling-layout.html) · 本书《从约束推导 AI Infra》

# 第 26 章 · 单个算子：分块、布局与切分

配套讲义：仓库里 `ai-infra/notes/lessons/26-tiling-layout.md`（附录最后一节"代码对照"）

**怎么用**：从上往下，一格一格按 **Shift + Enter** 运行。文字格是讲解，代码格是对应的 Python 写法；数字随便改，改完再按一次 Shift + Enter 看结果怎么变。纯 Python，不需要 GPU，也不下载模型。练习题不在这里算，先在讲义里自己做。

## 讲义 ↔ 代码

先看这张表：讲义里的每一步，对应下面哪一个函数、跑出来是什么。

| 讲义 | 公式 | 写法 | 跑出来 |
|---|---|---|---|
| 第二步 局部存储 | $2mk + 2kn + 4mn$ | `tile_local_kib(m, n)` | 8 / 24 / 80 KiB |
| 第二步 读写 | 重复读取 | `tile_traffic_mib(m, n)` | 6168 / 3096 / 1560 MiB |
| 第二步 驻留 | 100 KB ÷（工作集 + 1 KiB） | `resident(kib)` | 11 / 4 / 1 |
| 第三步 bank | 取模计数 | `bank_rounds(32)`、`bank_rounds(33)` | 32 / 1 轮 |
| 第四步 RMSNorm | | `rmsnorm_traffic_mib()` | 24 → 32.1 MiB |
| 第四步 次序 | FP32 舍入 | `f32(...)` | 差 1 ULP |

Qwen3-8B FFN 一支投影 [1024,4096]×[4096,12288]、例 5-3、RTX PRO 6000 的共享内存驻留、bank 冲突、RMSNorm 拆分与 FP32 加法次序取自 AII 第 5 章 §5.2。

## 1. 先把要用的东西定义好

这几格只是定义，运行后没有输出也正常。

In [ ]:
import struct
MiB = 2**20
M, K, N = 1024, 4096, 12288

**函数 `f32`**：舍入到 FP32。

In [ ]:
def f32(x: float) -> float:
    """舍入到 FP32。"""
    return struct.unpack("f", struct.pack("f", x))[0]

**函数 `tile_local_kib`**：一组输入缓冲 + FP32 累加器：2mk + 2kn + 4mn 字节。

In [ ]:
def tile_local_kib(m: int, n: int, k: int = 32) -> float:
    """一组输入缓冲 + FP32 累加器：2mk + 2kn + 4mn 字节。"""
    return (2 * m * k + 2 * k * n + 4 * m * n) / 1024

**函数 `tile_traffic_mib`**：各输出块独立读输入：(A 重复读取, W 重复读取, C 写出)，单位 MiB。

In [ ]:
def tile_traffic_mib(m: int, n: int) -> tuple[float, float, float]:
    """各输出块独立读输入：(A 重复读取, W 重复读取, C 写出)，单位 MiB。"""
    return 2 * M * K * (N / n) / MiB, 2 * K * N * (M / m) / MiB, 2 * M * N / MiB

**函数 `resident`**：小工具。

In [ ]:
def resident(kib: float, smem: int = 102_400, reserve: int = 1024) -> int:
    return smem // (int(kib * 1024) + reserve)

**函数 `bank_rounds`**：32 个 lane 读同一列（第 0 列）时，同一个 bank 最多被请求几次 = 要几轮。

In [ ]:
def bank_rounds(stride_words: int, lanes: int = 32, banks: int = 32, swizzle: bool = False) -> int:
    """32 个 lane 读同一列（第 0 列）时，同一个 bank 最多被请求几次 = 要几轮。"""
    hits: dict[int, int] = {}
    for r in range(lanes):
        col = (0 ^ r) if swizzle else 0
        b = (r * stride_words + col) % banks
        hits[b] = hits.get(b, 0) + 1
    return max(hits.values())

**函数 `rmsnorm_traffic_mib`**：(一组一行, 拆成 parts 段) 的总访问量：读输入 + 读 gamma + 写输出；拆分后多读一遍输入，再加局部和与缩放因子。

In [ ]:
def rmsnorm_traffic_mib(rows: int = 1024, width: int = 4096, parts: int = 8) -> tuple[float, float]:
    """(一组一行, 拆成 parts 段) 的总访问量：读输入 + 读 gamma + 写输出；拆分后多读一遍输入，再加局部和与缩放因子。"""
    x = rows * width * 2 / MiB
    one = 3 * x
    extra = (rows * parts * 4 * 2 + rows * 4 * 2) / MiB
    return one, one + x + extra

## 2. 一步一步跑

每格对应讲义里的一步，打印的文字说明了在算什么。

In [ ]:
print(f"\n第一步：{M}×{K} × {K}×{N}，BF16")
print(f"  运算 {2 * M * K * N / 1e9:.0f} GFLOPs；三份矩阵各一次 {2 * (M * K + K * N + M * N) / MiB:.0f} MiB")
print("\n第二步：例 5-3，k = 32、单组输入缓冲")
for t in (32, 64, 128):
    a, w, c = tile_traffic_mib(t, t)
    tot = a + w + c
    print(f"  {t:3d}×{t:<3d}：局部 {tile_local_kib(t, t):3.0f} KiB，访问 {a:.0f} + {w:.0f} + {c:.0f} = {tot:.0f} MiB，"
          f"强度 {2 * M * K * N / (tot * MiB):.1f}，100 KB 里驻留 {resident(tile_local_kib(t, t))} 块")
print(f"  大块只拿到四成带宽：时间比 {1560 / 3096 / 0.4:.2f}")

In [ ]:
print("\n第三步：32×32 FP32 暂存块，32 个 lane 读一列")
print(f"  行跨度 32：{bank_rounds(32)} 轮；补成 33：{bank_rounds(33)} 轮（空间 +{(33 / 32 - 1) * 100:.1f}%）；swizzle：{bank_rounds(32, swizzle=True)} 轮")

In [ ]:
print("\n第四步：RMSNorm 拆分与加法次序")
one, split = rmsnorm_traffic_mib()
print(f"  一组一行 {one:.0f} MiB；拆成 8 段 {split:.1f} MiB，第一阶段任务 1024 → 8192")
e = 2.0**-24
left, right = f32(f32(1 + e) + e), f32(1 + f32(e + e))
print(f"  FP32：(1 + 2^-24) + 2^-24 = {left!r}，1 + (2^-24 + 2^-24) = {right!r}，差 {(right - left) / 2**-23:.0f} ULP")

## 3. 代码里才看得到的

1. 64×64 的算术强度代码得 31.75，原文写 31.7（截断而非四舍五入）。
2. `f32` 用 `struct.pack("f")` 把 Python 的双精度数舍入到 FP32，不依赖 numpy。
3. swizzle 那行 `col = 0 ^ r` 看起来多余，换成读第 c 列时就是 `c ^ r`，读哪一列都是 1 轮。

## 4. 自己改一改

- 给 `tile_traffic_mib` 加 L2 复用：相邻 4 个列块共享一次 A 读取，总量变成多少？
- 用 `bank_rounds` 试行跨度 34、48、64，哪些跨度不冲突？规律是什么？
- 用 `f32` 做 4096 个随机数的平方和，比较 1 段、8 段、32 段的结果差几个 ULP。

在下面的空格子里写，或者直接改上面的格子。

In [ ]:
# 在这里写你的代码

## 口诀

> 手里这行还没放回，下一个输出就不用再取；
> 台面越大跑得越少，台子多了才能轮着等料；
> 都排一个窗口办三十二轮，补一格就一轮；
> 八个人分着数快，最后要加小计，还得再过一遍手，次序一换尾数就变。